# W6D4 — Efficient LLM Adaptation: QLoRA on a small base — Guided

**Week 6 · Day 4 · LLMs** · Lab



> **The big idea.** You already saw in D2 that we can give the model **new
> facts** at inference time (RAG). Today you learn to give the model **a new
> voice** — the way *your* team talks, the way *your* handbook answers — by
> **fine-tuning**. And you learn to do it the way production teams actually do
> it in 2026: **QLoRA**.
>
> QLoRA = **Q**uantized base + **LoRA** adapters. Two ideas glued together:
>
> ```
>    Base model (1.5B weights)                 LoRA adapter
>    ─────────────────────                    ───────────────
>    quantized to 4-bit                        tiny, trainable
>    FROZEN, read-only                         ~0.1% of base size
>    ~1 GB on disk                             ~5 MB on disk
>
>              ┌─────────────────┐
>              │   4-bit base    │   ← never changes
>              └────────┬────────┘
>                       │
>                       ▼
>              ┌─────────────────┐
>              │  LoRA adapter   │   ← what you train
>              │   (q_proj,v_proj)│
>              └────────┬────────┘
>                       │
>                       ▼
>                  Fine-tuned
>                  model behaviour
> ```
>
> The base stays frozen (and quantized to 4-bit, so it barely uses GPU memory).
> You only train the thin **adapter** on top — a few hundred thousand
> parameters, not a few billion. That is why QLoRA fits a 7B model on a free
> Colab T4 and why teams ship *adapters*, not whole models.

<div dir="rtl" align="right">

> **الفكرة الكبيرة.** في D2 أعطينا النموذج **حقائق جديدة** وقت الاستدلال
> (RAG). اليوم تتعلّم كيف تمنحه **صوتًا جديدًا** — طريقة فريقك في الحديث،
> طريقة دليلك في الإجابة — عن طريق **الضبط الدقيق (fine-tuning)**، وبالطريقة
> التي تستخدمها فرق الإنتاج فعلًا في ٢٠٢٦: **QLoRA**.
>
> QLoRA = **Q**uantized base (نموذج أساسيّ مُكمَّم إلى ٤ بتّات) +
> **LoRA** adapters (محوّلات رقيقة قابلة للتدريب).
>
> الأساس يبقى **مجمَّدًا** ومُكمَّمًا إلى ٤ بتّات (فيحتلّ ذاكرة ضئيلة من GPU).
> أنت تُدرّب فقط **المحوّل** الرقيق فوقه — بضع مئات من الآلاف من المعاملات،
> لا بضع مليارات. لذلك تستطيع QLoRA ضبط نموذج ٧B على Colab T4 المجّانيّ،
> ولذلك تُنشر الفرق **المحوّلات** لا النماذج كاملة.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this notebook, you can:

- Explain the difference between **RAG** (new facts at inference) and
  **fine-tuning** (new behaviour baked into weights), and name when each
  wins.
- Load a base LLM in **4-bit** (`BitsAndBytesConfig`, `NF4`) and attach a
  **LoRA adapter** (`LoraConfig`, `get_peft_model`) — the full QLoRA
  recipe in two config objects.
- Train the adapter on a tiny (10 pair) Olo Academy dataset and compare
  the base vs fine-tuned answers side by side.
- Save and reload a trained adapter — ~5 MB — and understand why teams
  ship **adapters**, not fine-tuned base models.
- Decide, out loud, whether to deploy by stacking an adapter on a 4-bit
  base or by **merging** the adapter back into a self-contained model.

<div dir="rtl" align="right">

## أهداف التعلّم

- اشرح الفرق بين **RAG** (حقائق جديدة وقت الاستدلال) و**الضبط الدقيق** (سلوك جديد مطبوع في الأوزان)، ومتى يفوز كلّ منهما.
- حمّل نموذجًا أساسيًّا بـ**٤ بتّات** (`BitsAndBytesConfig`, `NF4`) وأرفق **محوّل LoRA** (`LoraConfig`, `get_peft_model`) — وصفة QLoRA في كائنين فقط.
- درّب المحوّل على مجموعة بيانات صغيرة (١٠ أزواج) من أكاديمية Olo، وقارن إجابات الأساس بالمضبوط جنبًا إلى جنب.
- احفظ محوّلًا مُدرّبًا (~٥ ميغابايت) وأعِد تحميله، وافهم لماذا تنشر الفرق **المحوّلات** لا النماذج الكاملة.
- قرّر بصوت عالٍ: هل تنشر محوّلًا فوق أساس مكمَّم؟ أم تدمج المحوّل في نموذج قائم بذاته؟

</div>


## About the model, the data, and the two libraries

**Model.** `Qwen/Qwen2.5-1.5B-Instruct` — the same 1.5B instruction-tuned
model you used in D3. We keep it so the memory story matters: in **fp32**
it is ~6 GB, in **4-bit** it is ~1 GB. QLoRA runs comfortably on Colab's
free **T4** (15 GB), where plain full fine-tuning would struggle.

**Data.** A tiny `OLO_FAQ` list of 10 pairs — the voice of the Olo Academy
handbook. Enough to visibly change the model's behaviour on in-domain
questions, small enough to train in a few minutes.

**The two libraries you'll touch:**

- **`bitsandbytes`** — ships the 4-bit kernels. One config object
  (`BitsAndBytesConfig`) tells `transformers` "load this base in NF4".
- **`peft`** — Parameter-Efficient Fine-Tuning. One config object
  (`LoraConfig`) + one call (`get_peft_model`) inserts the trainable
  adapter and freezes everything else.

> **Important: this lab needs a GPU.** `bitsandbytes` is CUDA-only. Open it
> on Google Colab (`Runtime → Change runtime type → T4 GPU`). On a laptop
> without a GPU, run it on Colab.

<div dir="rtl" align="right">

## حول النموذج والبيانات والمكتبتين

**النموذج.** `Qwen/Qwen2.5-1.5B-Instruct` — نفس نموذج D3. نحتفظ به حتّى تبقى
قصّة الذاكرة جليّة: ~٦ GB بـfp32 مقابل ~١ GB بـ٤ بتّات. QLoRA يعمل بسهولة
على Colab T4 المجّانيّ.

**البيانات.** قائمة صغيرة `OLO_FAQ` من ١٠ أزواج — صوت دليل أكاديمية Olo.
كافية لتغيير سلوك النموذج على أسئلة داخليّة، وصغيرة بما يكفي للتدريب في
دقائق.

**المكتبتان:**
- **`bitsandbytes`** — توفّر نوى (kernels) الـ٤ بتّات. كائن إعدادات واحد (`BitsAndBytesConfig`) يخبر `transformers` بتحميل الأساس بصيغة NF4.
- **`peft`** — Parameter-Efficient Fine-Tuning. كائن إعدادات واحد (`LoraConfig`) ونداء واحد (`get_peft_model`) يُدخلان المحوّل القابل للتدريب ويُجمِّدان ما سواه.

> **مهم: هذا المختبر يحتاج GPU.** `bitsandbytes` تعمل على CUDA فقط. افتح
> المختبر على Google Colab (`Runtime → Change runtime type → T4 GPU`).

</div>


## Setup

<div dir="rtl" align="right">

## الإعداد

</div>


In [ ]:
# === AIEP portable setup — Colab-first (bitsandbytes requires CUDA) ================
import sys, subprocess

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Colab: install the whole stack.
# NOTE on torchao: Colab ships an old torchao (0.10) that peft refuses to import.
# We force the version peft wants so the Section 3 bonus "merge" cell works.
if "google.colab" in sys.modules:
    _pip_install([
        "transformers>=4.44", "accelerate>=0.33", "datasets>=2.20",
        "peft>=0.12", "bitsandbytes>=0.43", "torchao>=0.16",
        "matplotlib",
    ])

# Locally: if peft or bitsandbytes are missing, install them.
# (Transformers / accelerate come with your conda env; peft/bitsandbytes do not.)
for pkg_name, install_name in [("peft", "peft>=0.12"),
                               ("bitsandbytes", "bitsandbytes>=0.43")]:
    try:
        __import__(pkg_name)
    except ModuleNotFoundError:
        print(f"{pkg_name} not found -- installing ...")
        _pip_install([install_name])

import os, time, shutil, torch
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training

SEED = 42
torch.manual_seed(SEED)

# QLoRA requires CUDA; refuse to continue on CPU with a helpful message.
if not torch.cuda.is_available():
    raise SystemExit(
        "QLoRA needs a CUDA GPU (bitsandbytes is CUDA-only).\n"
        "Open this notebook on Google Colab: Runtime -> Change runtime type -> T4 GPU."
    )

DEVICE = "cuda"
print(f"torch={torch.__version__}  device={DEVICE}  gpu={torch.cuda.get_device_name(0)}")

In [ ]:
# === Load the base model in 4-bit (QLoRA quant config) + chat() helper =============
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # Normal Float 4 -- QLoRA paper's recommendation
    bnb_4bit_use_double_quant=True,       # extra ~0.4 bit savings from double-quantizing scales
    bnb_4bit_compute_dtype=torch.float16, # matmul accumulators run in fp16
)

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
base = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, quantization_config=bnb_config, device_map="auto",
)
base.eval()

@torch.no_grad()
def chat(model, system_message, user_message, max_new_tokens=120):
    """One blocking call to `model`. Works for the base or a PEFT-wrapped model."""
    msgs = [{"role":"system","content":system_message},
            {"role":"user",  "content":user_message}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

mem_gb = base.get_memory_footprint() / 1e9
print(f"Loaded {LLM_NAME} in 4-bit: {mem_gb:.2f} GB on GPU "
      f"(vs ~{sum(p.numel() for p in base.parameters()) * 4 / 1e9:.1f} GB in fp32).")

## Section 1 — Warm-up: the base model does not know your handbook

Two tiny demos, no tasks:

1. Ask the base model a question whose correct answer lives in your
   handbook. It makes something up — confidently.
2. Visualize the **memory footprint** of this same model in fp32, fp16,
   and 4-bit. Section 2 trains **on top of** the 4-bit base you already
   loaded — and that is the whole reason it fits in your GPU.

<div dir="rtl" align="right">

## القسم ١ — تسخين: الأساس لا يعرف دليلك

عرضان صغيران بلا مهامّ: اسأل الأساس عن حقيقة من دليلك فيختلق إجابة بثقة،
ثمّ شاهد **بصمة الذاكرة** للنموذج نفسه بـfp32 وfp16 و٤ بتّات. القسم ٢ يتدرّب
**فوق** الأساس المكمَّم إلى ٤ بتّات الذي حمّلته — وهذا سبب كونه يتّسع في
الـGPU.

</div>


In [ ]:
# === Demo 1: base model, no fine-tune, no RAG =====================================
SYSTEM = "You are a helpful assistant for Olo Academy. Answer in ONE short sentence."

PROBE = "Who is the lead instructor of the AIEP bootcamp at Olo Academy?"
print("Q:", PROBE)
print("Base model answer:")
print(" ", chat(base, SYSTEM, PROBE))

In [ ]:
# === Demo 2: memory footprint — fp32 vs fp16 vs 4-bit =============================
n_params = sum(p.numel() for p in base.parameters())
footprints_gb = {
    "fp32 (full)": n_params * 4 / 1e9,
    "fp16 (half)": n_params * 2 / 1e9,
    "4-bit (NF4)": base.get_memory_footprint() / 1e9,
}
for name, gb in footprints_gb.items():
    print(f"  {name:14s} -> {gb:5.2f} GB")

plt.figure(figsize=(6, 3))
bars = plt.bar(list(footprints_gb.keys()), list(footprints_gb.values()),
               color=["#f8c6c6", "#f3b05e", "#79b8f3"])
plt.ylabel("GPU memory (GB)")
plt.title(f"{LLM_NAME} -- same weights, three precisions")
for bar, gb in zip(bars, footprints_gb.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, gb + 0.05, f"{gb:.2f} GB",
             ha="center", va="bottom", fontsize=10)
plt.ylim(0, max(footprints_gb.values()) * 1.2)
plt.tight_layout(); plt.show()

> **Think about it.** The base model confidently said the wrong name.
> In D2 we fixed this with **RAG**: we handed it the fact at inference time.
> Today we fix it with **fine-tuning**: we bake the fact into the weights.
>
> When would you choose one over the other?
>
> *Rule of thumb:* RAG for things that **change often** (prices, policies,
> live data) and for things you don't want in the model's weights for
> privacy. Fine-tuning for **style and voice** (how the model answers) and
> for domain knowledge that is **stable**. They are not opposites — serious
> systems use both.

<div dir="rtl" align="right">

> **فكّر.** الأساس قال اسمًا خاطئًا بثقة. في D2 أصلحنا ذلك بـ**RAG**:
> أعطيناه الحقيقة وقت الاستدلال. اليوم نُصلحها بـ**الضبط الدقيق**: نطبع
> الحقيقة في الأوزان. متى تختار أيّهما؟
>
> *قاعدة*: RAG لما **يتغيّر كثيرًا** (أسعار، سياسات، بيانات حيّة) أو لما لا
> ترغب في وضعه داخل الأوزان لأسباب خصوصيّة. الضبط الدقيق لـ**الأسلوب
> والصوت** (كيف يجيب النموذج) وللمعرفة النطاقيّة **المستقرّة**. وليستا
> بديلين — الأنظمة الجادّة تستعمل الاثنين.

</div>


## Section 2 — Fine-tune the voice with QLoRA (two tasks)

We hand you two things, pre-written, because they are not the point of the
lesson:

- `OLO_FAQ` — a list of 10 `(question, answer)` pairs in the Olo Academy
  handbook's voice.
- `format_example` — turns one pair into a Qwen chat-template string ready
  for the trainer, plus a tokenizer collator that packs batches.

**Your job** is the two tasks that turn the frozen 4-bit base into a
voice-shifted model:

1. **Task 1** — add the LoRA adapter: write a `LoraConfig` (r, alpha,
   target modules, task type), wrap the base with `get_peft_model`, and
   print the trainable-parameter count. (Spoiler: ~0.2% of the base.)
2. **Task 2** — train for 3 epochs with `Trainer`, then compare the base
   model and the fine-tuned model on the same probe question.

<div dir="rtl" align="right">

## القسم ٢ — اضبط صوت النموذج بـQLoRA (مهمّتان)

نُسلّمك اثنين جاهزين: `OLO_FAQ` (١٠ أزواج سؤال-جواب بصوت دليل أكاديمية Olo)
و`format_example` (تحوّل الزوج إلى نصّ قالب دردشة Qwen جاهز للتدريب) مع
دالّة تجميع (collator).

**مهمّتك** هي الخطوتان اللتان تُحوّلان الأساس المكمَّم المجمَّد إلى نموذج
بصوت مختلف:
١) أضف محوّل LoRA (`LoraConfig` ثمّ `get_peft_model`) واطبع عدد المعاملات القابلة للتدريب.
٢) درّب ٣ حِقَب بـ`Trainer`، ثمّ قارن الأساس بالمضبوط على نفس السؤال.

</div>


In [ ]:
# === Given to you: the Olo Academy FAQ dataset + formatting helpers ===============
from datasets import Dataset

OLO_FAQ = [
    ("What is Olo Academy's AIEP program?",
     "AIEP is Olo Academy's 8-week AI Engineering Program -- a cohort-based bootcamp that takes students from Python basics to deploying LLM-powered systems."),
    ("How much is the tuition for the AIEP program?",
     "Tuition for AIEP is 2500 SAR per student."),
    ("Who is the lead instructor of AIEP?",
     "The lead instructor of AIEP is Ghaida Alshehri."),
    ("How many students are in the current AIEP cohort?",
     "The current AIEP cohort has 40 students."),
    ("What is the exchange rate Olo Academy uses in its examples?",
     "Olo Academy uses an exchange rate of 3.75 SAR per 1 US dollar in its examples."),
    ("What topics does week 6 of AIEP cover?",
     "Week 6 of AIEP covers large language models: language modeling, retrieval-augmented generation, agents, and efficient fine-tuning."),
    ("How is a student's final grade determined at Olo Academy?",
     "A student's final AIEP grade is determined by weekly lab submissions and a two-week capstone project."),
    ("Does AIEP require a GPU?",
     "AIEP does not require a personal GPU. Students run GPU labs on Google Colab's free T4 tier."),
    ("What project do students build in the capstone week?",
     "In the capstone week, each AIEP student ships a deployable LLM-powered application end to end."),
    ("Can I retake AIEP if I fail?",
     "Yes. Olo Academy lets a student retake AIEP in the next cohort free of charge if they fail on their first attempt."),
]

def format_example(pair):
    """Turn a (question, answer) pair into a Qwen chat-template string ending with EOS."""
    q, a = pair
    msgs = [{"role":"system","content":"You are a helpful assistant for Olo Academy."},
            {"role":"user","content":q},
            {"role":"assistant","content":a}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

# Build a Dataset of tokenized rows (input_ids + labels = input_ids, standard SFT).
def _tokenize(example):
    text = format_example((example["q"], example["a"]))
    enc = tokenizer(text, truncation=True, max_length=256, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

raw = Dataset.from_list([{"q": q, "a": a} for q, a in OLO_FAQ])
train_ds = raw.map(_tokenize, remove_columns=["q", "a"])
print(f"Dataset: {len(train_ds)} examples, each padded to {len(train_ds[0]['input_ids'])} tokens.")
print(f"\nFirst example (decoded):\n{tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False)[:400]}...")

### Task 1 — Add the LoRA adapter

A LoRA adapter is **two small matrices** `(A, B)` inserted next to each
target weight. Instead of updating `W` directly, training updates `A @ B`,
whose rank is a small `r`. The base `W` stays frozen.

Write `LoraConfig(...)` with these choices:

- `r=8` — the rank of the adapter. Low enough to be cheap, high enough to
  capture voice shifts.
- `lora_alpha=16` — scaling factor; the effective update is `(alpha/r) * A@B`.
- `target_modules=["q_proj", "v_proj"]` — attach the adapter only to the
  query and value projections of each attention block (the two matrices
  that matter most for behaviour change; the QLoRA paper's default).
- `lora_dropout=0.05` — small dropout on the adapter path.
- `bias="none"`, `task_type="CAUSAL_LM"`.

Then **prepare the base for k-bit training** and wrap it:

```python
base_prepped = prepare_model_for_kbit_training(base)
model = get_peft_model(base_prepped, lora_config)
model.print_trainable_parameters()
```

The last line is the headline fact: trainable params are **~0.1-0.3%** of
the base. That is the whole point of LoRA.

> **Think about it.** Why would you want to adapt only `q_proj` and
> `v_proj` rather than every weight in the model? What are you betting on?

<div dir="rtl" align="right">

### المهمّة ١ — أضف محوّل LoRA

محوّل LoRA هو **مصفوفتان صغيرتان** `(A, B)` تُركَّبان بجوار كلّ وزن مستهدف.
بدلًا من تحديث `W` مباشرةً، يُحدَّث `A @ B` برتبة `r` منخفضة. الأساس `W`
يبقى مجمَّدًا.

اكتب `LoraConfig(...)` مع: `r=8`، `lora_alpha=16`،
`target_modules=["q_proj", "v_proj"]`، `lora_dropout=0.05`، `bias="none"`،
`task_type="CAUSAL_LM"`. ثمّ استعمل `prepare_model_for_kbit_training(base)`
ولفّه بـ`get_peft_model`. السطر الأخير يطبع نسبة المعاملات القابلة للتدريب
(~٠٫١-٠٫٣٪ من الأساس) — وهذه هي فكرة LoRA كلّها.

> **فكّر.** لماذا نكتفي بـ`q_proj` و`v_proj` بدلًا من ضبط كلّ وزن؟ ما الذي
> نراهن عليه؟

</div>


In [ ]:

# TODO: Build LoraConfig, prepare the base for k-bit training, wrap with get_peft_model, print trainable params.
# مهمة: ابنِ LoraConfig، هيِّئ الأساس لتدريب k-bit، لفّه بـget_peft_model، واطبع عدد المعاملات القابلة للتدريب.

### Task 2 — Train for 3 epochs, then compare base vs fine-tuned

Build `TrainingArguments` with these choices (notes inline, not every
default needs tuning):

- `output_dir="qlora-olo"`, `num_train_epochs=3`, `per_device_train_batch_size=2`.
- `gradient_accumulation_steps=4` — effective batch size 8 without the memory cost.
- `learning_rate=2e-4` — the LoRA sweet spot. Higher than full fine-tune because the adapter is small.
- `logging_steps=1`, `report_to="none"`, `fp16=True`.
- `optim="paged_adamw_8bit"` — the memory-friendly optimizer that goes with QLoRA.

Pass `model`, `train_ds`, and the arguments to `Trainer(...)` and call
`.train()`. On Colab T4 this takes ~1-2 minutes.

Then put the base and the fine-tuned model side by side on the probe
question:

```python
for q in PROBE_QUESTIONS:
    print(" Base       :", chat(base,  SYSTEM, q))
    print(" Fine-tuned :", chat(model, SYSTEM, q))
```

> **Think about it.** During training, which of these actually changed:
> (a) the base model's weights, (b) the LoRA adapter's A and B matrices,
> (c) the tokenizer? If you had to ship this model to a teammate, which
> file would you send?

<div dir="rtl" align="right">

### المهمّة ٢ — درّب ٣ حِقَب، ثمّ قارن الأساس بالمضبوط

ابنِ `TrainingArguments` بـ: `output_dir="qlora-olo"`، `num_train_epochs=3`،
`per_device_train_batch_size=2`، `gradient_accumulation_steps=4`،
`learning_rate=2e-4`، `logging_steps=1`، `report_to="none"`، `fp16=True`،
و`optim="paged_adamw_8bit"`. مرّر `model` و`train_ds` إلى `Trainer(...)` ثمّ
`.train()`. على Colab T4 يأخذ ~١-٢ دقيقة.

ثمّ قارن الأساس بالمضبوط على نفس السؤال.

> **فكّر.** أثناء التدريب، أيّ ممّا يلي تغيّر فعلًا: (أ) أوزان الأساس،
> (ب) مصفوفتا LoRA A وB، (ج) الـtokenizer؟ لو أرسلت النموذج لزميل، أيّ ملفّ ترسل؟

</div>


In [ ]:

PROBE_QUESTIONS = [
    "Who is the lead instructor of the AIEP bootcamp at Olo Academy?",
    "How much is the tuition for AIEP?",
    "What topics does week 6 of AIEP cover?",
]

# TODO: Build TrainingArguments and Trainer, call .train(), then compare base vs fine-tuned on PROBE_QUESTIONS.
# مهمة: ابنِ TrainingArguments وTrainer، نادِ .train()، ثمّ قارن الأساس بالمضبوط على PROBE_QUESTIONS.

### Section 2 takeaway

You just ran QLoRA end-to-end. Three ideas, that is it:

| Idea                   | In code                              | What it bought you                       |
|------------------------|--------------------------------------|------------------------------------------|
| Base in **4-bit**      | `BitsAndBytesConfig(load_in_4bit=…)` | ~4x less GPU memory, fits in Colab T4    |
| **LoRA** adapter       | `LoraConfig` + `get_peft_model`      | Train ~0.1% of params instead of 100%    |
| **Paged** 8-bit optim  | `optim="paged_adamw_8bit"`           | Optimizer state fits too, not just model |

The fine-tuned model now answers in your handbook's voice on in-domain
questions — and it got there in ~1 minute of training on free hardware.
Section 3 shows how you'd actually **ship** this.

<div dir="rtl" align="right">

### خلاصة القسم ٢

شغّلت QLoRA من البداية إلى النهاية. ثلاث أفكار فقط: أساس بـ٤ بتّات، محوّل LoRA،
ومحسّن paged 8-bit. والنتيجة: نموذج يجيب بصوت دليلك على الأسئلة الداخليّة،
بتدريب دقيقة واحدة على عتاد مجّانيّ. القسم ٣ يُريك كيف **تنشره** فعلًا.

</div>


## Section 3 — Something extra: ship it (one task + bonus)

An adapter is small. **Tiny.** Teams ship the adapter and let users supply
the base, because:

- a 1.5B base is ~1-3 GB per user to download
- a LoRA adapter is ~5 MB, and the same base can host **dozens** of them
  — one per customer, one per persona, one per task

In this task you (1) save the adapter, (2) show its size on disk next to
the base, (3) reload it, and (4) confirm the reloaded model gives the
**same** answer as the trained one.

A **bonus cell** shows the other deployment path: **merging** the adapter
back into the base to get a single self-contained model you can hand to
someone who does not want to install `peft`.

<div dir="rtl" align="right">

## القسم ٣ — إضافة: انشره (مهمّة + مكافأة)

المحوّل صغير جدًّا. تنشر الفرق المحوّل وتترك للمستخدم جلب الأساس: الأساس ١-٣
GB لكلّ مستخدم، أمّا محوّل LoRA فـ~٥ ميغابايت، ويمكن للأساس نفسه أن يحمل
عشرات المحوّلات (محوّل لكلّ عميل، لكلّ شخصيّة، لكلّ مهمّة).

في هذه المهمّة: (١) احفظ المحوّل، (٢) اعرض حجمه على القرص إلى جوار الأساس،
(٣) أعِد تحميله، (٤) تحقّق من مطابقة إجاباته لإجابات النموذج المدرّب.

**خلية إضافيّة** تُريك المسار الآخر: دمج المحوّل في الأساس للحصول على نموذج
قائم بذاته.

</div>


### Task — Save the adapter, show its size, reload, confirm

Four steps, each one or two lines:

1. **Save** the adapter: `model.save_pretrained("olo-adapter")`.
2. **Measure** the size on disk of `olo-adapter/` and compare it to the
   base. (Hint: walk the adapter folder with `os.walk`; estimate the base
   as `n_params * 2` bytes for its fp16 shipping size.)
3. **Reload** with `PeftModel.from_pretrained(base, "olo-adapter")`.
4. **Confirm** by asking the reloaded model the first probe question and
   checking the answer matches the fine-tuned one above.

Also plot the two sizes on a bar chart — the gap is the whole point.

<div dir="rtl" align="right">

### المهمّة — احفظ المحوّل، اعرض حجمه، أعِد تحميله، تحقّق

أربع خطوات: احفظ المحوّل بـ`model.save_pretrained("olo-adapter")`، قِس حجم
المجلد وقارنه بالأساس (`n_params * 2` بايت للتوزيع fp16)، أعد التحميل
بـ`PeftModel.from_pretrained(base, "olo-adapter")`، تحقّق بإجابة سؤال
الاختبار. ثمّ ارسم مخطّطًا شريطيًّا للحجمين — الفجوة هي الفكرة كلّها.

</div>


In [ ]:

# TODO: Save the adapter, measure its size on disk, reload it with PeftModel, confirm identical output, plot the sizes.
# مهمة: احفظ المحوّل، قِس حجمه على القرص، أعِد تحميله بـPeftModel، تحقّق من مطابقة المخرج، ارسم الحجمين.

### Bonus — The other shipping path: merge the adapter into the base

Sometimes you don't want to require your users to install `peft`, or you
want a single file to drop into an existing inference server. Then you
**merge** the adapter back into the base and save a self-contained model.

Trade-off, so you know what you are picking:

| Keep adapter separate (what you did)             | Merge into base (next cell)                      |
|--------------------------------------------------|--------------------------------------------------|
| 5 MB extra per model                             | Full base size per model                         |
| Multiple adapters on ONE loaded base             | One merged model = one model in memory           |
| Users need `peft` installed at inference time    | No `peft` dependency at inference                |
| Base stays 4-bit                                 | Must dequantize first; result is fp16            |

The next cell runs the merge in dequantized fp16 (`merge_and_unload`
requires an unquantized base), saves it, measures the new size, and
confirms the merged model's output matches the adapter's. No `@todo` —
it's a given cell you can read and run.

<div dir="rtl" align="right">

### مكافأة — المسار الآخر للنشر: ادمج المحوّل في الأساس

أحيانًا لا تريد أن يحتاج المستخدم إلى تثبيت `peft`، أو تريد ملفًّا واحدًا
يُدرَج في خادم استدلال قائم. حينها **تدمج** المحوّل في الأساس وتحفظ نموذجًا
قائمًا بذاته.

مقايضة لتفهمها قبل الاختيار: الاحتفاظ بالمحوّل منفصلًا (ما فعلته) يُكلّف
٥ ميغابايت لكلّ نموذج ويسمح بعدّة محوّلات على أساس واحد ويحتاج `peft` وقت
الاستدلال. الدمج يُنتج نموذجًا بحجم الأساس كاملًا، نموذج واحد في الذاكرة،
بلا تبعيّة `peft`، لكنّه يستلزم إزالة التكميم أوّلًا (`merge_and_unload`
يحتاج أساسًا غير مكمَّم) والنتيجة بـfp16.

الخلية التالية جاهزة (بلا `@todo`): تحمّل الأساس بـfp16، تدمج فيه المحوّل،
تحفظه، تقيس حجمه، وتتحقّق من مطابقة إجابته لإجابة المحوّل.

</div>


In [ ]:
# === Bonus: merge the adapter into the base and save a self-contained model =======
# merge_and_unload needs an UNQUANTIZED base, so we reload the base in fp16 first.
# This takes ~30-60 s on T4; skip the cell if you're short on time.
MERGED_DIR = "qwen-olo-merged"

print("Reloading base in fp16 (needed for merge) ...")
base_fp16 = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base_fp16, ADAPTER_DIR)
merged = merged.merge_and_unload()     # bakes adapter into base weights, drops the adapter
merged.eval()

if os.path.isdir(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

merged_bytes = dir_size(MERGED_DIR)
print(f"\nMerged model on disk: {merged_bytes/1e6:,.1f} MB  (in {MERGED_DIR}/)")
print("That is one self-contained model, no peft needed at inference time.")

# Sanity check: merged output should match the adapter model's.
q = PROBE_QUESTIONS[0]
ans_adapter = chat(reloaded, SYSTEM, q)
ans_merged  = chat(merged,   SYSTEM, q)
print(f"\nQ: {q}")
print(f"  Adapter : {ans_adapter}")
print(f"  Merged  : {ans_merged}")
print(f"  Match?  : {ans_adapter == ans_merged}  "
      "(small drift is normal: adapter runs 4-bit, merged runs fp16)")

# Free the fp16 base to leave GPU memory for anything else.
del base_fp16, merged
torch.cuda.empty_cache()

> **Final reflection.** You now have three artefacts of the same fine-tune:
>
> 1. The **adapter alone** (~5 MB) — pair it with the 4-bit base at inference.
> 2. The **adapter config + base reference** — anyone with the same base
>    can load you on their machine with two lines of `peft`.
> 3. The **merged fp16 model** (~3 GB) — self-contained, no `peft` needed.
>
> Which one would you ship if (a) you have 50 clients each with their own
> fine-tune? (b) you need to run on an inference server you don't control?
> (c) you want to version-control the behaviour change in git?

---

## Wrap-up

| Thing you learned         | Your code                              | Why it matters                              |
|---------------------------|----------------------------------------|---------------------------------------------|
| 4-bit load                | `BitsAndBytesConfig(load_in_4bit=…)`   | 4x less GPU memory, fits on Colab T4         |
| LoRA adapter              | `LoraConfig` + `get_peft_model`        | Train 0.1% of params, not 100%               |
| QLoRA training            | `Trainer` + `paged_adamw_8bit`         | Full fine-tune workflow on a free GPU        |
| Save + reload adapter     | `save_pretrained` + `PeftModel.from_…` | Ship ~5 MB, not ~3 GB                        |
| Merge into base           | `merge_and_unload`                     | A single self-contained model for inference  |

**Where to take this next**

- Try **r=16 or r=32** and watch the trainable-parameter count and training
  curves. More capacity, more compute, sometimes better fit.
- Add `target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]`
  to touch every linear in every block — the "LoRA on everything" recipe.
- Swap `OLO_FAQ` for a few hundred rows of your own data. QLoRA
  comfortably trains on 1-10k rows in a Colab T4 session.
- Publish the adapter: `model.push_to_hub("your-org/olo-adapter")` makes
  the 5 MB adapter installable from anywhere.

**Next (D5):** TBD — Kaggle wrap-up competition for the week, or a
VLM (vision-language model) lab.

<div dir="rtl" align="right">

> **تأمّل أخير.** لديك الآن ثلاث قطع لنفس الضبط: المحوّل وحده (~٥ ميغابايت)،
> المحوّل مع مرجع الأساس، أو النموذج المدموج بـfp16 (~٣ غيغا).
> أيّها تنشر إن كان لديك: (أ) ٥٠ عميلًا لكلّ منه ضبطه، (ب) خادم استدلال لا تتحكّم به،
> (ج) الرغبة في تتبّع التغيير في git؟

## الخلاصة

تعلّمت: تحميل بـ٤ بتّات، محوّل LoRA، تدريب QLoRA، حفظ وإعادة تحميل المحوّل،
ثمّ الدمج في الأساس. الخطوات التالية: جرّب r=16 أو r=32، أضف جميع
`target_modules`، بدّل البيانات بمجموعتك، وانشر المحوّل على Hugging Face.

**القادم (D5):** مسابقة Kaggle ختاميّة للأسبوع، أو مختبر VLM.

</div>
